<a href="https://colab.research.google.com/github/JamieMIP/forage-jpmc-swe-task-1/blob/main/menu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Five Security Issues

1 - Non encrypted personal data, none of the txts are encrypted despite containing data about what customers have been doing

2 - No password protection, nothing to ensure the person using the system is actually allowed to be

3 - No logs are kept of changes made to the data. Not only does this mean its hard to fix if a mistake is made, but makes it impossible to trace the actions of anyone using the system maliciously and fix the damage they may have done

4 - When data is removed from a file it can still be found in the file history, this could mean potentially sensitive data believed to be removed is still accessable

5 - If a file is deleted from the drive, there is no backup to recover it from. This makes is suseptable to attack such as ransomware, in which the data is held hostage

#File linking **important**
When using this replace base path with the folder structure containing the file. e.g. My Drive/Coursworks/Marking.

This is essential so the code can access files from google drive.

*This was written by F534550 beggining 12/01/2026*

In [ ]:
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import clear_output

from google.colab import drive
import os

drive.mount('/content/drive')
base_path = 'My Drive' # <------ Change folder structure here

# function to take any file in the project and attach the file structure to the start
def get_path(base_path, file_name):
  base_path = os.path.join('/content/drive', base_path)
  base_path = os.path.join(base_path, 'Programming_Coursework')
  return(os.path.join(base_path, file_name))




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Key imports:

This cell contains the imports and file paths for subscriptions and feedbacks along with giving me access to the date.

*This was written by F534550 beggining 13/01/2026*

In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL

# set the file paths to use with smSL and fmSL
feedback_path = get_path(base_path, "Game_Feedback.txt")
subscription_path = get_path(base_path, "Subscription_Info.txt")

#feedback = fmSL.load_feedback(feedback_path) # (for debugging)
#print(feedback) # (for debugging)


from datetime import date, timedelta
today = str(date.today())

#Home button that can be used anywhere:

*This was written by F534550 beggining 12/01/2026*

In [ ]:

go_to_main_menu_button = widgets.Button(
    description = 'main menu'
)

#Searching games:

Enter the search query into the box. Use the toggle box to select weather you are looking for a video or board game. Use dropdown menu to choose what you want to search by e.g. name or genre.

*This was written by F534550 beggining 11/01/2026*

In [ ]:
label = widgets.Label(value="Search Games:")

# drop down menu defining what to search by
# the options array is subject to change depending on weather it is searching for board or video games
search_settings = widgets.Dropdown(
 description = '   Search by: ',
 options = ['Name', 'Platform', 'Genre'],
 value = 'Name'
)

# defines the search box widget
search_box = widgets.Text(
  value='',
  placeholder = 'Type to search...',
  disabled = False,
  layout = widgets.Layout(width='400px')
)

# a toggle to switch between searching video and board games
binary_toggle = widgets.ToggleButton(
    value = False,
    description = 'Video Games'
)

# updating toggle name and search options to match board or video games
def on_toggle(change):
    global file_name
    if change['new']:
        binary_toggle.description = 'Board Games'
        search_settings.options = ['Name', 'Player Count', 'Genre']

    else:
        binary_toggle.description = 'Video Games'
        search_settings.options = ['Name', 'Platform', 'Genre']


binary_toggle.observe(on_toggle, names='value')

# this box will hold the outputs of our search results

results_box = widgets.VBox(
        layout = widgets.Layout(
        border = '4px solid black',
        max_height = '400px',
        width = '500px',
        overflow = 'auto'
        )
      )

# triggered when enter pressed in searchbox, responsable for finding and displaing results
def enter_search(sender): # sender perameter passes in the object that called it
    element = search_settings.options.index(search_settings.value) + 1
    captured_text = sender.value
    # we must convert the status of the toggle into the correct filepath by taking the first 5 letters (video or board)
    # then add on the ending "_Game_Info.txt" which is universal to both files
    games = get_path(base_path, binary_toggle.description[:5] + '_Game_Info.txt')
    fetched_data = []

    # access whatever file is held in games (either video_game_info.txt or board_game_info.txt)
    with open(games, 'r') as f:
      lines = f.readlines()
      # iterating through the file and selecting all rows that contain the searched text
      for row in (lines):
        print(row.split(",")[element])
        if captured_text.upper() in row.split(",")[element].upper():
          fetched_data.append(row.strip()) # array with all data relevent to the search

      # search results are dispalyed as a series of labels within a box
      output = []
      for data in (fetched_data):
        # atatching weather the game is available from rental.txt
        with open(get_path(base_path, 'Rental.txt'), 'r') as rental_file:
          available = True
          lines = rental_file.readlines()
          for row in (lines):
            # finding matching gameIDs between
            if row[:5] == data[:5] and row.split(',')[2] == ' ':
              available = False

        # add available/not available accordinly to the end of each line
        if available:
          data = data + ', Available'
        else:
          data = data + ', Not Available'

        # for each relevent game a new label is created which contains a row from the csv file
        # the first 3 digits are cut to conver the UID e.g. 'min01' into an index '01'
        new_label = widgets.Label(value = data[3:])
        output.append(new_label)

      clear_output(wait=True) # clears the screen (prevents bug where multiple search results appear at the same time)
      search_game()
      results_box.children = output
      display(results_box)

search_box.on_submit(enter_search) # when enter is pressed runs the enter_search() function

search_box_container = widgets.VBox([
    widgets.HBox([
        widgets.HBox([label, search_box, binary_toggle]),
        go_to_main_menu_button
    ], layout=Layout(justify_content='space-between', width='100%')),
    search_settings
])



# when this functon is run the search page appears
def search_game(button = 0): # button perameter is mandatory if it is called by a button, otherwise it uses 0 to prevent error
  clear_output(wait=True)
  display(search_box_container)

#Renting Games:

Enter the customer and game ID and press submit, it will tell you weather it was valid or invalid.

*This was written by F534550 beggining 13/01/2026*

In [ ]:
# enter_customer_ID, enter_game_ID, submit_rent_button, rent_a_game_box
# are the main widgets for the renting games menu

enter_customer_ID = widgets.Text(
    padding = "60px",
    description = 'Customer ID',
    value='',
    placeholder = 'Type to search...',
    disabled = False,
    layout=Layout(width='400px')
)

enter_game_ID = widgets.Text(
  padding = "60px",
  description = 'Game ID',
  value='',
  placeholder = 'Type to search...',
  disabled = False,
  layout=Layout(width='400px')
)

submit_rent_button = widgets.Button(
  description='Submit',
  layout=Layout(width='80px', height='25px')
)

rent_a_game_box = widgets.VBox([
    widgets.HBox([
        enter_customer_ID, go_to_main_menu_button], layout=Layout(justify_content='space-between', width='100%')),
      enter_game_ID, submit_rent_button
])

# activated when submit button is hit
# responsible for prosessing weather a rental is valid and then performing it
def validate_rental(button):
  customer_ID = enter_customer_ID.value
  game_ID = enter_game_ID.value
  game_exists = False

  # stores all of the subscriptions from Subscription_Info.txt
  subscriptions = smSL.load_subscriptions(subscription_path)

  # checks if the subscription is invalid
  if not smSL.check_subscription(customer_ID, subscriptions):
    #print("invalid") # (for debugging)
    output_label = widgets.Label(value = 'subscription invalid')

  # runs if the subscription is valid
  else:
    with open(get_path(base_path, 'Rental.txt'), 'r') as rental_file:
      lines = rental_file.readlines()
      for row in (lines):
        #print(row[:5], row.split(',')[2]) # (for debugging)

        # checks if there is no return date,
        # meaning the game is currently being rented out
        if row[:5] == game_ID and row.split(',')[2] == ' ':
          #print("game already rented") # (for debugging)
          output_label = widgets.Label(value = 'game already rented')
        else:
          # we assume the game doens't exist untill it's found
          game_exists = False

          # firstly check all of the board games for the query
          with open(get_path(base_path, 'Board_Game_Info.txt'), 'r') as bg:
            lines_bg = bg.readlines()

            for row_bg in (lines_bg):
              if row_bg[:5] == game_ID:
                game_exists = True

          # secondly check all of the video games for the query
          with open(get_path(base_path, 'Video_Game_Info.txt'), 'r') as vg:
            lines_vg = vg.readlines()

            for row_vg in (lines_vg):
              if row_vg[:5] == game_ID:
                game_exists = True

          # output an error if the game doesn't exist
          if not game_exists:
            #print("game is not in inventory") # (for debugging)
            output_label = widgets.Label(value = 'game is not in inventory')

      if game_exists:
        #print("game rented sucsesfully") # (for debugging)
        output_label = widgets.Label(value = 'game rented sucsesfully')

        # create a new row for rental with no return date
        new_row = [game_ID, today, ' ', customer_ID]

        # add the new row into Rental.txt
        with open(get_path(base_path, 'Rental.txt'), 'a') as rental_file:
          #print("adding new rental") # (for debugging)
          rental_file.write(','.join(new_row) + '\n')

  display(output_label)

# link submit button to its functionality
submit_rent_button.on_click(validate_rental)

# displays game renting GUI
def rent_a_game(button = 0): # button perameter is mandatory if it is called by a button, otherwise it uses 0 to prevent error
  clear_output(wait=True)
  display(rent_a_game_box)

# for manually populating rental.txt
  # with open(get_path(base_path, 'Rental.txt'), 'a') as rental_file:
#     print("adding new rental") # (for debugging)
#     rental_file.write(','.join(new_row) + '\n')




#Returning games

Enter customer and game ID and press submit, then use the dropdown menu to select a star rating and add any additional feedback in the text box.

*This was written by F534550 beggining 13/01/2026*

In [ ]:

# submit_return_button, submit_feedback_button, enter_additional_feedback, star_rating
# are the main widgets elements for the returning games menu
submit_return_button = widgets.Button(
  description='Submit',
  layout=Layout(width='80px', height='25px')
)

submit_feedback_button = widgets.Button(
    description='Submit',
    layout=Layout(width='80px', height='25px')
)

enter_additional_feedback = widgets.Text(
    value='',
    placeholder = 'Additional feedback',
    disabled = False,
    layout=Layout(width='300px')
)

star_rating = widgets.Dropdown(
    options = ['5', '4', '3', '2', '1'],
    value = '5',
    layout=Layout(width='50px')
)

# used to collect feedback and add it to Game_Feedback.txt
def submit_feedback(button): # One perameter minimum is mandatory
  # fetch the star rating and additonal feedback
  star = int(star_rating.value)
  additional_feedback = enter_additional_feedback.value

  # update Game_Feedback.txt
  fmSL.add_feedback(enter_game_ID.value, star, additional_feedback, feedback_path)

# activates when the submit_return_button is hit
def validate_return(button):
  rental_found = False

  # fetch customer and game ID from the search boxes
  customer_ID = enter_customer_ID.value
  game_ID = enter_game_ID.value

  with open(get_path(base_path, 'Rental.txt'), 'r') as rental_file:
      # by default we assume that the return won't work and update if its found
      output_label = widgets.Label(value = 'game return failed')
      lines = rental_file.readlines()

      # checks for an unavailable game that matches the ID
      for i, row in enumerate(lines):
        if row[:5] == game_ID and row.split(',')[2] == ' ' and row[-5:-1] == customer_ID:
          output_label = widgets.Label(value = 'game returned successfully')

          # store the updated line with the return date added in and the number of the updated line
          update_line = ','.join([row.split(',')[0], row.split(',')[1], today, row.split(',')[3]])
          update_line_num = i
          rental_found = True

  if rental_found == True:
    # if the rental is valid the entire file is stored in lines and the updated line is applied
    lines[update_line_num] = update_line + '\n'

    # re write all the values to the fiel
    with open(get_path(base_path, 'Rental.txt'), 'w') as f:
      f.writelines(lines)

  # return an error message if the rental isn't found
  else:
    output_label = widgets.Label(value = 'game return failed')

  # reset the GUI
  clear_output(wait=True)


  display(widgets.HBox([
    widgets.HBox([star_rating, enter_additional_feedback, submit_feedback_button]),
    go_to_main_menu_button], layout=Layout(justify_content='space-between', width='100%')
    ))

  display(output_label)

# displaying all the widgets
# reuseing the search boxes: enter_customer_ID and enter_game_ID from game rentals
return_a_game_box = widgets.VBox([
    widgets.HBox([
        enter_customer_ID, go_to_main_menu_button], layout=Layout(justify_content='space-between', width='100%')),
      enter_game_ID, submit_return_button
])

# linking submit buttons to there functionality
submit_return_button.on_click(validate_return)
submit_feedback_button.on_click(submit_feedback)

# Displays game return GUI
def return_a_game(button = 0): # One perameter minimum is mandatory
  clear_output(wait=True)
  display(return_a_game_box)

#Inventory Pruning:

You can use the buttons to see either the 3 worst rated games or the 3 least popular and then enter a game id in the textbox to remove it

*This was written by F534550 beggining 13/01/2026*

In [ ]:
# you can either prune by the number of times a game is rented
# or by the average rating that game recived on return
# these two buttons reflect that
prune_by_populatiry = widgets.Button(
    description = 'three lowest popularity',
    layout = Layout(width = '300px')
)

prune_by_rating = widgets.Button(
    description = 'three lowest rating',
    layout = Layout(width = '300px')
)

# text box for inputing a game to remove
remove_box = widgets.Text(
    description = 'remove game',
    placeholer = 'ID...',
    disabled = False,
    layout=Layout(width='200px')
)

# function to find average of a list, used to calculate average star rating
def find_average(list_in): # perameter is the list which average is calculated from
  total = 0
  for item in (list_in):
    total = total + item
  return round(total/len(list_in), 2)

# combines two equal length list into a 2d array
def combine_lists(list_1, list_2): # each peramiter is one of the lists
  new_list = []
  for i in range (len(list_1)):
    new_list.append([list_1[i], list_2[i]])
  return new_list

# returns 3 worst rated games
def rating_prune(button): # one perameter minimum
  prune() # prevents results from stacking if multiple button presses
  # we need a list for each unique game
  # and a list of scores that matches that
  games = []
  scores = []

  # first we get a list of all the games that have ratings
  feedback = fmSL.load_feedback(feedback_path) # gives dictionary with keys GameID, Ratings, Comments
  for row in (feedback):
    if row['GameID'] not in games:
      games.append(row['GameID'])

  # next we collect an average rating scoree for each game (out of 5 stars)
  for game in (games):
    temp_scores = []

    for row in (feedback):
      if row['GameID'] == game:
        temp_scores.append(row['Rating'])
        # the average scores are then added to a list that mirrors games
    scores.append(find_average(temp_scores))


  # combining the lists
  games_scores = combine_lists(games, scores)
  # sort by counts (second element)
  games_scores.sort(key=lambda row: row[1])
  # take the 3 least popular items
  games_scores = games_scores[:3]

  # displaying the ratings as labels in a VBox
  labels = [widgets.Label(value = "Game : Times Rented")]
  for game in (games_scores):
    labels.append(
    widgets.Label(value=f"{game[0]} : {game[1]}")  # f string required as game is a string and scores[i] float
    )

  ratings_box = widgets.VBox(labels)
  display(ratings_box)

# outputs 3 least popular games
def popularity_prune(button): # one perameter minimum
  prune() # prevents results from stacking if multiple button presses
  # we need a list for each unique game
  # and a list of how many times that game has been rented
  games = []
  counts = []

  # first we get all the board games
  with open(get_path(base_path, 'Board_Game_Info.txt'), 'r') as f:
    lines = f.readlines()
    for row in (lines):
      if row.split(',')[0] not in games:
        games.append(row.split(',')[0])

  # then all the video games
  with open(get_path(base_path, 'Video_Game_Info.txt'), 'r') as f:
    lines = f.readlines()
    for row in (lines):
      if row.split(',')[0] not in games:
        games.append(row.split(',')[0])

  # counting up how many times each game has been rented (includes not yet returned)
  for game in (games):
    count = 0
    with open(get_path(base_path, 'Rental.txt'), 'r') as f:
      lines = f.readlines()
      for row in (lines):
        if row.split(',')[0] == game:
          count = count + 1

    counts.append(count)

  # combining the lists
  games_counts = combine_lists(games, counts)
  # sort by counts (second element)
  games_counts.sort(key=lambda row: row[1])
  # take the 3 least popular items
  games_counts = games_counts[:3]

  # displaying the counts as labels in a VBox
  labels = [widgets.Label(value = "Game : Times Rented")]
  for game in (games_counts):
    labels.append(
    widgets.Label(value=f"{game[0]} : {game[1]}")  # f string required as game is a string and scores[i] float
    )

  popularity_box = widgets.VBox(labels)
  display(popularity_box)

prune_box = widgets.HBox([widgets.HBox([prune_by_populatiry, prune_by_rating, remove_box]),
            go_to_main_menu_button], layout=Layout(justify_content='space-between', width='100%')
)

# linking buttons to there functionality
prune_by_rating.on_click(rating_prune)
prune_by_populatiry.on_click(popularity_prune)

# Removes a signle line from a file
# perameters are the file to remove from, the contents of the file and the line being removed
def remove_row(file_name, lines, index):
    del lines[index]

    with open(file_name, 'w', newline='') as f:
        f.writelines(lines)

# Removes a game from a single file
# perameters are the game ID and the file to remove from
def remove_item(game, file_name):
  file_name = get_path(base_path, file_name)

  with open(file_name, 'r') as f:
    lines = f.readlines()
    for i, row in enumerate(lines):

      if row.split(",")[0] == game:
        remove_row(file_name, lines, i)

# Activates on pressing enter
# Removes a game from all files
def remove_game(sender): # perameter passes in the object that called the function
  game_ID = sender.value
  to_remove_from = ["Board_Game_Info.txt", "Video_Game_Info.txt", "Rental.txt", "Game_Feedback.txt"]
  for file_name in (to_remove_from):
    remove_item(game_ID, file_name)
  display(label(value = "Game remove successful (may take a minute to sync)"))


remove_box.on_submit(remove_game)

# displays the prune GUI
def prune(button = 0): # One perameter mandatory
  clear_output(wait=True)
  display(prune_box)

#Bookings:

Enter the user ID, if it is valid you will then get a selection of dropdown boxes. Use them to select the date, time and number of guests and click submit.
If there are to many bookings on that slot it will let you know and not make the booking.

*This was written by F534550 beggining 13/01/2026*

In [ ]:
# Dropdown menues book_slots, book_time, book_date, book_guests and there submit button

book_slots = widgets.Text(
    description = 'enter ID',
    placeholer = 'ID...',
    disabled = False,
    layout=Layout(width='300px')
)

book_time = widgets.Dropdown(
    options = ['2pm', '6pm'],
    value = '2pm',
    layout=Layout(width='150px')
)

# store the next 10 days in an array
next_ten_days = []
for i in range(10):
  next_ten_days.append(str(date.today() + timedelta(days=i)))

book_date = widgets.Dropdown(
    options = next_ten_days,
    value = today,
    layout=Layout(width='150px')
)

book_guests = widgets.Dropdown(
    options = ['0', '1', '2', '3'],
    value = '0',
    layout=Layout(width='150px')
)

options_submit_button = widgets.Button(
    description = 'submit',
    layout = Layout(width = '150px')
)


# Display all booking options along with their labels, the submit button and the home button
booking_options_box = widgets.VBox([widgets.HBox([widgets.HBox([
      widgets.Label(value = 'time'),
      widgets.Label(value = 'date', layout=Layout(padding='0px 0px 0px 130px')),
      widgets.Label(value = 'guest count', layout=Layout(padding='0px 0px 0px 125px'))
      ]),
    go_to_main_menu_button], layout=Layout(justify_content='space-between', width='100%')
), widgets.HBox([book_time, book_date, book_guests, options_submit_button])
])

# called when enter is pressed in the text box
# checks weather a booking user has a valid subscription and then transitions to the options GUI
def book_slot(sender):
  customer_ID = sender.value
  # stores all of the subscriptions from Subscription_Info.txt
  subscriptions = smSL.load_subscriptions(subscription_path)

  # checks if the subscription is invalid
  if not smSL.check_subscription(customer_ID, subscriptions):
    output_label = widgets.Label(value = 'subscription invalid')

  # if subscription is valid bring up all the options for a booking
  else:
    # clear gui and display options
    clear_output(wait=True)
    display(booking_options_box)

# activated when the options are submitted using the submit button
# collects the options, checks if the booking is full then updates Bookings.txt
def options_submit(button):
  # get customer ID
  customer_ID = book_slots.value
  # get option values
  date = book_date.value
  time = book_time.value
  guests = int(book_guests.value)

  # checking if there are 50 people booked that day (including desks)
  with open(get_path(base_path, 'Bookings.txt'), 'r') as f:
    lines = f.readlines()

    # holds total number of people booked including pending booking
    booked = guests + 1

    for row in (lines):
      # check if a booking has the same time and date
      if row.split(",")[1] == date and row.split(",")[2] == time:
          booked = booked + 1 + int(row.split(",")[3])

    if booked <= 50:
      new_row = f'{customer_ID},{date},{time},{guests}\n'

      # add the new row into Rental.txt
      with open(get_path(base_path, 'Bookings.txt'), 'a') as rental_file:
        rental_file.write(new_row)

      output_label = widgets.Label(value = 'booking sucessfull')
    else:
      output_label = widgets.Label(value = 'booking failed, too many people')

    display(output_label)

# links book_slot button to it's functionality
book_slots.on_submit(book_slot)

# links the submit button to it's functionality
options_submit_button.on_click(options_submit)

# displays booking GUi
def booking(button): # One perameter mandatory
  clear_output(wait=True)
  booking_box = widgets.HBox([widgets.HBox([book_slots]),
            go_to_main_menu_button], layout=Layout(justify_content='space-between', width='100%')
  )

  display(booking_box)



###Main menu:

There are five buttons for searching games, renting games, returning, games, pruning games and booking a session. Simply click the relevent button.

*This was written by F534550 beggining 07/01/2026*

In [ ]:
### This was written by F534550 beggining 07/01/2026 ###

def main_menu(button): # One perameter mandatory
  clear_output(wait=True)
  main_menu_buttons_layout = Layout(
      # This sets the layout properties for all major buttons on the main menu
      width = "18%",
      height = "40%",
      padding = "6px",
      #margin = "0 25px 0 25px" #hard coded px margin encase % doesn't work
      # Note - margin follows up right down left
      margin = "0 3% 0 3%"
  )

  main_menu_buttons_style = {
      # Like the main_menu_buttons_layout this standerdises the styles across buttons in the main menu
      # Note - uses CSS
      "font_weight": "bold"
  }

  # main_menu_title = widgets.Label(
  #     value = "Main Menu",
  #     layout = Layout(
  #         margin = "0 0 20px 0",
  #         width = "100%",
  #         align_items = "center",
  #         justify_content = "center"
  #                     ),
  #     style={"font_weight": "bold"}
  # )
  main_menu_title = widgets.HTML(
      value="<h2 style='text-align:center; font-weight:bold; margin-bottom:20px;'>Main Menu</h2>"
  )

  search_games_button = widgets.Button(
      # Button linking to search game functionality
      description = "search games",
      layout = main_menu_buttons_layout,
      style = main_menu_buttons_style
  )

  rent_game_button = widgets.Button(
      # Button linking to rent game functionality
      description = "rent game",
      layout = main_menu_buttons_layout,
      style = main_menu_buttons_style
  )

  book_button = widgets.Button(
      # Button linking to booking functionality
      description = "book",
      layout = main_menu_buttons_layout,
      style = main_menu_buttons_style
  )

  return_games_button = widgets.Button(
      # Button linking to returning games functionality
      description = "return games",
      layout = main_menu_buttons_layout,
      style = main_menu_buttons_style
  )

  pruning_button = widgets.Button(
      # Button linking to pruning functionality
      description = "pruning",
      layout = main_menu_buttons_layout,
      style = main_menu_buttons_style
  )

  # Function calls for button clicks
  search_games_button.on_click(search_game)
  rent_game_button.on_click(rent_a_game)
  return_games_button.on_click(return_a_game)
  pruning_button.on_click(prune)
  book_button.on_click(booking)

  box = widgets.VBox([
        main_menu_title,
        # Dislays all main navigation buttons horizontally
        widgets.HBox([
            search_games_button,
            rent_game_button,
            book_button,
            return_games_button,
            pruning_button

        ], layout = Layout(justify_content = "center")) # Centers buttons
  ])

  display(box)

# Functionality for the main menu button (has to be done after the main menu)

go_to_main_menu_button.on_click(main_menu)
main_menu(0) # Starting the code on the main menu
#print(uploaded.keys())
#!ls




VBox(layout=Layout(border='4px solid black', max_height='400px', overflow='auto', width='500px'))